In [0]:
%sql
create connection if not exists eq_project_connection_test
type HTTP
OPTIONS (
  host = 'https://earthquake.usgs.gov',
  port = 443,
  base_path = '/earthquakes/feed/v1.0/',
  bearer_token = 'na'
)

In [0]:
## getting the connection detail in dynamic 
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

conn = w.connections.get("eq_project_connection_test")
#print(conn)
base_url = f"{conn.options['host']}{conn.options['base_path']}"
print(base_url)

In [0]:
## the catalog name will be come in dynamic way, becase this code will be deploy on prod, and now we are on dev
dbutils.widgets.text("catalog_name", "earthquake_proj","earthquake_proj")
catalog_name = dbutils.widgets.get("catalog_name")
print(catalog_name)

In [0]:
%sql
-- use the correct syntax for catalog and schema selection, and fix volume creation
-- USE CATALOG earthquake_proj;
-- USE SCHEMA bronze;
-- CREATE VOLUME IF NOT EXISTS bronze_volume;



In [0]:
## for dynamic catalog 
spark.sql(
    f"use catalog {catalog_name}"
)
spark.sql(
    "use schema bronze"
)
spark.sql(
    "create volume if not exists bronze_volume"
)

In [0]:
import requests

#url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
## will take the url from base_url varialbe
## this is getting status=400 error why?
import datetime
import json
url = f'{base_url}summary/all_day.geojson'
response = requests.get(url)

if response.status_code == 200:
    try:
        data = response.json()
        current_date=datetime.datetime.now().strftime("%Y-%m-%d")
       # print(data)
       # write into the bronze_volume
        dbutils.fs.put(f"/Volumes/{catalog_name}/bronze/bronze_volume/eq_bronze_volume{current_date}.json",
                        json.dumps(data), True)
    except Exception as e:
        print("Invalid JSON:", e)
        print(response.text[:500])
else:
    print(f"Request failed: {response.status_code}")
    print(response.text[:500])